In [ ]:
from pyspark.sql.functions import avg, col, max, min, stddev

silver_table = spark.table("nyc_taxi.silver.green_taxi")
outlier_table = spark.table("nyc_taxi.quarantine.taxi_numerical_outliers_latest")
total_records = silver_table.count()
print(f"DATA QUALITY REPORT - nyc_taxi.silver.green_taxi")
print(f"Total Records: {total_records:,}")

checks = []

def add_check(name, violations):
    checks.append({"name": name, "violations": int(violations)})

# Completeness checks.
critical_fields = [
    "trip_id", "vendor_id", "pickup_datetime", "dropoff_datetime",
    "PULocationID", "DOLocationID", "trip_distance", "total_amount",
]
for field in critical_fields:
    add_check(f"Null values: {field}", silver_table.filter(col(field).isNull()).count())

# Uniqueness and validity checks.
add_check("Duplicate trip IDs", total_records - silver_table.select("trip_id").distinct().count())
add_check("Negative fares", silver_table.filter(col("fare_amount") < 0).count())
add_check("Negative total amounts", silver_table.filter(col("total_amount") < 0).count())
add_check("Zero trip distance", silver_table.filter(col("trip_distance") == 0).count())
add_check("Trip duration over 24 hours", silver_table.filter(col("trip_duration_minutes") > 1440).count())
add_check("Non-positive trip duration", silver_table.filter(col("trip_duration_minutes") <= 0).count())
add_check("Invalid pickup locations", silver_table.filter((col("PULocationID") < 1) | (col("PULocationID") > 265)).count())
add_check("Invalid dropoff locations", silver_table.filter((col("DOLocationID") < 1) | (col("DOLocationID") > 265)).count())
add_check("Quarantined numerical outliers", outlier_table.count())

# Transformation audit counts.
for label, field in [
    ("Passenger count imputed", "passenger_count_was_imputed"),
    ("Payment type imputed", "payment_type_was_imputed"),
    ("Rate code imputed", "rate_code_was_imputed"),
    ("Reversed/refund records", "is_reversed"),
]:
    if field in silver_table.columns:
        print(f"{label}: {silver_table.filter(col(field) == True).count():,}")

print("Numerical outliers by reason:")
outlier_table.groupBy("outlier_reason").count().orderBy(col("count").desc()).show(truncate=False)

for check in checks:
    status = "PASS" if check["violations"] == 0 else "WARN"
    print(f"{status}: {check['name']} = {check['violations']:,}")

stats = silver_table.select(
    avg("fare_amount").alias("avg_fare"),
    stddev("fare_amount").alias("stddev_fare"),
    min("fare_amount").alias("min_fare"),
    max("fare_amount").alias("max_fare"),
    avg("trip_distance").alias("avg_distance"),
    max("trip_distance").alias("max_distance"),
    avg("trip_duration_minutes").alias("avg_duration"),
    max("trip_duration_minutes").alias("max_duration"),
).first().asDict()

print(f"Fare average: ${stats['avg_fare'] or 0:.2f}; max: ${stats['max_fare'] or 0:.2f}")
print(f"Distance average: {stats['avg_distance'] or 0:.2f}; max: {stats['max_distance'] or 0:.2f}")
print(f"Duration average: {stats['avg_duration'] or 0:.2f}; max: {stats['max_duration'] or 0:.2f}")

passed_checks = sum(check["violations"] == 0 for check in checks)
total_checks = len(checks)
quality_score = passed_checks / total_checks * 100 if total_checks else 0
print(f"Overall Quality Score: {quality_score:.1f}% ({passed_checks}/{total_checks} checks passed)")